In [1]:
import cv2
import torch
import torch.nn.functional as F
from models.src.encoder import YOLODataEncoder
from models.src.yolo import YOLO
from models.utils.data import load_groundtruths

/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
input_size=(300, 300)

In [3]:
data_path = "dataset/Dataset/validation/Vehicle registration plate"
data = load_groundtruths(data_path, train=False, shuffle=False, debug=True)
images, boxes, labels, tot_snamples = data
idx = 1
sample = (images[idx], boxes[idx], labels[idx])
sample

Total 386 images and 386 boxes loaded from: dataset/Dataset/validation/Vehicle registration plate
Debug mode enabled: Only using 10 samples for set: validation


('dataset/Dataset/validation/Vehicle registration plate/00723dac8201a83e.jpg',
 [[1.398784, 408.147456, 60.326912, 438.24460799999997],
  [787.3146879999999, 347.72966399999996, 830.90432, 374.35161600000004]],
 [1, 1])

In [4]:
image = cv2.imread(sample[0])
img_size = image.shape
img_size

(768, 1024, 3)

In [5]:
bboxes = sample[1]
bboxes_norm = torch.tensor(bboxes, dtype=torch.float)

bboxes_norm[:, [0, 2]] *= (input_size[1] / img_size[1])
bboxes_norm[:, [1, 3]] *= (input_size[0] / img_size[0])

In [6]:
yolo_model = YOLO(backbone_name= "resnet18",
                 train_backbone= False,
                 returned_layers= [4],
                 num_classes= 2,
                 fpn_channels= 256)

Freezing conv1.weight
Freezing bn1.weight
Freezing bn1.bias
Freezing layer1.0.conv1.weight
Freezing layer1.0.bn1.weight
Freezing layer1.0.bn1.bias
Freezing layer1.0.conv2.weight
Freezing layer1.0.bn2.weight
Freezing layer1.0.bn2.bias
Freezing layer1.1.conv1.weight
Freezing layer1.1.bn1.weight
Freezing layer1.1.bn1.bias
Freezing layer1.1.conv2.weight
Freezing layer1.1.bn2.weight
Freezing layer1.1.bn2.bias
Freezing layer2.0.conv1.weight
Freezing layer2.0.bn1.weight
Freezing layer2.0.bn1.bias
Freezing layer2.0.conv2.weight
Freezing layer2.0.bn2.weight
Freezing layer2.0.bn2.bias
Freezing layer2.0.downsample.0.weight
Freezing layer2.0.downsample.1.weight
Freezing layer2.0.downsample.1.bias
Freezing layer2.1.conv1.weight
Freezing layer2.1.bn1.weight
Freezing layer2.1.bn1.bias
Freezing layer2.1.conv2.weight
Freezing layer2.1.bn2.weight
Freezing layer2.1.bn2.bias
Freezing layer3.0.conv1.weight
Freezing layer3.0.bn1.weight
Freezing layer3.0.bn1.bias
Freezing layer3.0.conv2.weight
Freezing layer

/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
yolo_model.backbone.strides

[32]

In [8]:
image.shape

(768, 1024, 3)

In [9]:
image = cv2.resize(image, input_size)
image_tensor = torch.tensor(image, dtype=torch.float).permute(2, 0, 1).unsqueeze(0)
image_tensor.shape

torch.Size([1, 3, 300, 300])

In [10]:
out = yolo_model(image_tensor)

In [11]:
out.shape

torch.Size([1, 100, 7])

In [12]:
encoder = YOLODataEncoder(input_size=input_size, strides=yolo_model.backbone.strides)
encoded = encoder.encode(bboxes_norm, sample[2])
encoded.shape

torch.Size([100, 6])

In [13]:
num_classes = 2

labels = encoded[:, 5].long()

fake_class_logits = torch.full(
    (encoded.shape[0], num_classes),
    -10.0,
    device=encoded.device
)

fake_class_logits[torch.arange(encoded.shape[0]), labels] = 10.0
#
# fake_logits = torch.cat([encoded[:, :5], fake_class_logits], dim=1)
target_obj = encoded[:, 0]
fake_obj_logits = torch.where(target_obj > 0.5,
                              torch.full_like(target_obj, 10.0),
                              torch.full_like(target_obj, -10.0))

fake_bbox = encoded[:, 1:5]  # exact match -> bbox loss ~0
fake_logits = torch.cat([fake_obj_logits.unsqueeze(1), fake_bbox, fake_class_logits], dim=1)
fake_logits.shape

torch.Size([100, 7])

In [14]:
encoder.decode(fake_logits)

tensor([[230.6586, 135.8319, 243.4290, 146.2311,   0.7311,   1.0000],
        [  0.4098, 159.4326,  17.6739, 171.1893,   0.7311,   1.0000]])

In [15]:
bboxes_norm

tensor([[  0.4098, 159.4326,  17.6739, 171.1893],
        [230.6586, 135.8319, 243.4290, 146.2311]])

In [16]:
def yolo_loss(pred_logits, target_encoded, lambda_obj=1.0, lambda_bbox=1.0, lambda_cls=1.0):
    """
    Args:
        pred_logits: [B, N, 5 + num_classes] or [N, 5 + num_classes]
        target_encoded: [B, N, 6] or [N, 6]
    """
    # Flatten batch dimension if present
    if pred_logits.ndim == 3:
        B = pred_logits.shape[0]
        pred_logits = pred_logits.reshape(-1, pred_logits.shape[-1])
        target_encoded = target_encoded.reshape(-1, target_encoded.shape[-1])

    pred_obj = pred_logits[:, 0]
    pred_bbox = pred_logits[:, 1:5]
    pred_cls = pred_logits[:, 5:]

    target_obj = target_encoded[:, 0]
    target_bbox = target_encoded[:, 1:5]
    target_cls = target_encoded[:, 5].long()

    # 1. Objectness loss (binary, all cells)
    loss_obj = F.binary_cross_entropy_with_logits(pred_obj, target_obj)

    # 2. Bounding box regression (only positive cells)
    pos_mask = (target_obj > 0.5)
    if pos_mask.any():
        loss_bbox = F.smooth_l1_loss(pred_bbox[pos_mask], target_bbox[pos_mask])
    else:
        loss_bbox = torch.tensor(0.0, device=pred_logits.device)

    # 3. Class loss (only positive cells, cross-entropy on logits)
    if pos_mask.any():
        loss_cls = F.cross_entropy(pred_cls[pos_mask], target_cls[pos_mask])
    else:
        loss_cls = torch.tensor(0.0, device=pred_logits.device)

    total_loss = lambda_obj * loss_obj + lambda_bbox * loss_bbox + lambda_cls * loss_cls
    return total_loss

In [17]:
yolo_loss(fake_logits, encoded)

tensor(4.5769e-05)

In [18]:
print(out.shape, encoded.unsqueeze(0).shape)
print(fake_logits.shape, encoded.shape)

torch.Size([1, 100, 7]) torch.Size([1, 100, 6])
torch.Size([100, 7]) torch.Size([100, 6])


In [19]:
yolo_loss(fake_logits, encoded)

tensor(4.5769e-05)

In [20]:
import math
import torch
import torch.nn.functional as F

def sigmoid_focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    alpha: float = 0.25,
    gamma: float = 2.0,
    reduction: str = "mean",
):
    """
    Binary focal loss on logits.
    logits/targets shape: [M] or [M, ...], targets in {0,1}.
    """
    targets = targets.to(dtype=logits.dtype)
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    p_t = p * targets + (1.0 - p) * (1.0 - targets)
    alpha_t = alpha * targets + (1.0 - alpha) * (1.0 - targets)
    loss = alpha_t * (1.0 - p_t).pow(gamma) * bce

    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss


def ciou_loss_xyxy(pred_boxes: torch.Tensor, tgt_boxes: torch.Tensor, eps: float = 1e-7):
    """
    CIoU loss for aligned boxes in xyxy format.
    pred_boxes/tgt_boxes shape: [K, 4]
    """
    # Intersection
    x1 = torch.maximum(pred_boxes[:, 0], tgt_boxes[:, 0])
    y1 = torch.maximum(pred_boxes[:, 1], tgt_boxes[:, 1])
    x2 = torch.minimum(pred_boxes[:, 2], tgt_boxes[:, 2])
    y2 = torch.minimum(pred_boxes[:, 3], tgt_boxes[:, 3])

    inter_w = (x2 - x1).clamp(min=0)
    inter_h = (y2 - y1).clamp(min=0)
    inter = inter_w * inter_h

    # Areas + IoU
    pw = (pred_boxes[:, 2] - pred_boxes[:, 0]).clamp(min=eps)
    ph = (pred_boxes[:, 3] - pred_boxes[:, 1]).clamp(min=eps)
    tw = (tgt_boxes[:, 2] - tgt_boxes[:, 0]).clamp(min=eps)
    th = (tgt_boxes[:, 3] - tgt_boxes[:, 1]).clamp(min=eps)

    area_p = pw * ph
    area_t = tw * th
    union = area_p + area_t - inter + eps
    iou = inter / union

    # Centers
    px = (pred_boxes[:, 0] + pred_boxes[:, 2]) * 0.5
    py = (pred_boxes[:, 1] + pred_boxes[:, 3]) * 0.5
    tx = (tgt_boxes[:, 0] + tgt_boxes[:, 2]) * 0.5
    ty = (tgt_boxes[:, 1] + tgt_boxes[:, 3]) * 0.5

    # Enclosing box diagonal
    cx1 = torch.minimum(pred_boxes[:, 0], tgt_boxes[:, 0])
    cy1 = torch.minimum(pred_boxes[:, 1], tgt_boxes[:, 1])
    cx2 = torch.maximum(pred_boxes[:, 2], tgt_boxes[:, 2])
    cy2 = torch.maximum(pred_boxes[:, 3], tgt_boxes[:, 3])
    c2 = ((cx2 - cx1).pow(2) + (cy2 - cy1).pow(2)).clamp(min=eps)

    # Center distance
    rho2 = (px - tx).pow(2) + (py - ty).pow(2)

    # Aspect ratio consistency
    v = (4.0 / (math.pi ** 2)) * (torch.atan(tw / th) - torch.atan(pw / ph)).pow(2)
    alpha = v / (1.0 - iou + v + eps)

    ciou = iou - (rho2 / c2) - alpha * v
    return (1.0 - ciou).mean()


def yolo_loss_v2(
    pred_logits: torch.Tensor,         # [B,N,5+C] or [N,5+C]
    target_encoded: torch.Tensor,      # [B,N,6]   or [N,6]
    *,
    pred_boxes_xyxy: torch.Tensor = None,   # optional [B,N,4] or [N,4]
    target_boxes_xyxy: torch.Tensor = None, # optional [B,N,4] or [N,4]
    lambda_obj: float = 1.0,
    lambda_bbox: float = 5.0,
    lambda_cls: float = 1.0,
    use_focal_obj: bool = True,
    focal_alpha: float = 0.25,
    focal_gamma: float = 2.0,
):
    # Flatten batch if present
    if pred_logits.ndim == 3:
        pred_logits = pred_logits.reshape(-1, pred_logits.shape[-1])
        target_encoded = target_encoded.reshape(-1, target_encoded.shape[-1])
        if pred_boxes_xyxy is not None:
            pred_boxes_xyxy = pred_boxes_xyxy.reshape(-1, 4)
        if target_boxes_xyxy is not None:
            target_boxes_xyxy = target_boxes_xyxy.reshape(-1, 4)

    pred_obj = pred_logits[:, 0]
    pred_bbox = pred_logits[:, 1:5]      # encoded deltas
    pred_cls = pred_logits[:, 5:]

    tgt_obj = target_encoded[:, 0]
    tgt_bbox = target_encoded[:, 1:5]
    tgt_cls = target_encoded[:, 5].long()

    pos_mask = tgt_obj > 0.5

    # Objectness
    if use_focal_obj:
        loss_obj = sigmoid_focal_loss(pred_obj, tgt_obj, alpha=focal_alpha, gamma=focal_gamma)
    else:
        loss_obj = F.binary_cross_entropy_with_logits(pred_obj, tgt_obj)

    # Box
    if pos_mask.any():
        if pred_boxes_xyxy is not None and target_boxes_xyxy is not None:
            loss_bbox = ciou_loss_xyxy(pred_boxes_xyxy[pos_mask], target_boxes_xyxy[pos_mask])
        else:
            # Fallback on encoded deltas if decoded xyxy not available yet
            loss_bbox = F.smooth_l1_loss(pred_bbox[pos_mask], tgt_bbox[pos_mask])
        loss_cls = F.cross_entropy(pred_cls[pos_mask], tgt_cls[pos_mask])
    else:
        loss_bbox = pred_logits.new_tensor(0.0)
        loss_cls = pred_logits.new_tensor(0.0)

    total = lambda_obj * loss_obj + lambda_bbox * loss_bbox + lambda_cls * loss_cls
    return total, {"obj": loss_obj.detach(), "bbox": loss_bbox.detach(), "cls": loss_cls.detach()}

In [21]:
yolo_loss_v2(fake_logits, encoded)

(tensor(6.9875e-14),
 {'obj': tensor(6.9875e-14), 'bbox': tensor(0.), 'cls': tensor(0.)})

In [22]:
yolo_loss_v2(out, encoded.unsqueeze(0))

(tensor(14.3541, grad_fn=<AddBackward0>),
 {'obj': tensor(0.1327), 'bbox': tensor(2.7055), 'cls': tensor(0.6939)})

In [23]:
import torch
import torch.nn.functional as F


def sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0, reduction="mean"):
    targets = targets.to(dtype=logits.dtype)
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    p_t = p * targets + (1.0 - p) * (1.0 - targets)
    alpha_t = alpha * targets + (1.0 - alpha) * (1.0 - targets)
    loss = alpha_t * (1.0 - p_t).pow(gamma) * bce
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss


def softmax_focal_loss(logits, targets, alpha=0.25, gamma=2.0, reduction="mean"):
    # targets: class indices [K], logits: [K, C]
    ce = F.cross_entropy(logits, targets, reduction="none")
    pt = torch.exp(-ce)
    # Simple scalar alpha (same for all positive classes)
    at = torch.full_like(pt, alpha)
    loss = at * (1.0 - pt).pow(gamma) * ce
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss


def yolo_loss_v3(
    pred_logits,             # [B,N,5+C] or [N,5+C]
    target_encoded,          # [B,N,6] or [N,6]
    *,
    num_classes,
    obj_loss="focal",        # "focal" | "bce"
    cls_loss="focal",        # "focal" | "ce" | "bce"
    box_loss="s1",           # "s1" | "l1"
    lambda_obj=1.0,
    lambda_bbox=5.0,
    lambda_cls=1.0,
    focal_alpha=0.25,
    focal_gamma=2.0,
):
    if pred_logits.ndim == 3:
        pred_logits = pred_logits.reshape(-1, pred_logits.shape[-1])
        target_encoded = target_encoded.reshape(-1, target_encoded.shape[-1])

    pred_obj = pred_logits[:, 0]
    pred_bbox = pred_logits[:, 1:5]
    pred_cls = pred_logits[:, 5:5 + num_classes]

    tgt_obj = target_encoded[:, 0].to(pred_obj.dtype)
    tgt_bbox = target_encoded[:, 1:5].to(pred_bbox.dtype)
    tgt_cls = target_encoded[:, 5].long()

    pos_mask = tgt_obj > 0.5

    # Objectness
    if obj_loss == "focal":
        loss_obj = sigmoid_focal_loss(pred_obj, tgt_obj, alpha=focal_alpha, gamma=focal_gamma)
    elif obj_loss == "bce":
        loss_obj = F.binary_cross_entropy_with_logits(pred_obj, tgt_obj)
    else:
        raise ValueError(f"Unsupported obj_loss: {obj_loss}")

    # Box + Class only on positives
    if pos_mask.any():
        if box_loss == "s1":
            loss_bbox = F.smooth_l1_loss(pred_bbox[pos_mask], tgt_bbox[pos_mask])
        elif box_loss == "l1":
            loss_bbox = F.l1_loss(pred_bbox[pos_mask], tgt_bbox[pos_mask])
        else:
            raise ValueError(f"Unsupported box_loss: {box_loss}")

        if cls_loss == "focal":
            loss_cls = softmax_focal_loss(
                pred_cls[pos_mask], tgt_cls[pos_mask], alpha=focal_alpha, gamma=focal_gamma
            )
        elif cls_loss == "ce":
            loss_cls = F.cross_entropy(pred_cls[pos_mask], tgt_cls[pos_mask])
        elif cls_loss == "bce":
            tgt_onehot = F.one_hot(tgt_cls[pos_mask], num_classes=num_classes).to(pred_cls.dtype)
            loss_cls = F.binary_cross_entropy_with_logits(pred_cls[pos_mask], tgt_onehot)
        else:
            raise ValueError(f"Unsupported cls_loss: {cls_loss}")
    else:
        z = pred_logits.new_tensor(0.0)
        loss_bbox, loss_cls = z, z

    total = lambda_obj * loss_obj + lambda_bbox * loss_bbox + lambda_cls * loss_cls
    return total, {"obj": loss_obj.detach(), "bbox": loss_bbox.detach(), "cls": loss_cls.detach()}

In [24]:
yolo_loss_v3(fake_logits, encoded, num_classes=2)

(tensor(6.9875e-14),
 {'obj': tensor(6.9875e-14), 'bbox': tensor(0.), 'cls': tensor(0.)})

In [25]:
yolo_loss_v3(out, encoded.unsqueeze(0), num_classes=2)

(tensor(13.7037, grad_fn=<AddBackward0>),
 {'obj': tensor(0.1327), 'bbox': tensor(2.7055), 'cls': tensor(0.0434)})

In [26]:
from models.src.loss import YOLODetectionLoss

criterion = YOLODetectionLoss(num_classes=2)
criterion(out, encoded.unsqueeze(0))

(tensor(13.7037, grad_fn=<AddBackward0>),
 {'obj': tensor(0.1327), 'bbox': tensor(2.7055), 'cls': tensor(0.0434)})